[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/08_Kalman_Filter.ipynb)

# DiveLab

## Notebook 08 — Kalman Filtering

**From deterministic observers to probabilistic estimation**

### Guiding question

How should an estimator combine a dynamic model with noisy depth measurements when neither is perfectly trustworthy?

This notebook uses a local linear model and synthetic Gaussian uncertainty. It is an instructional experiment, not a model of certified dive-computer performance.

## Learning objectives

By the end of this notebook, you should be able to:

- construct a discrete stochastic state-space model;
- distinguish process covariance $Q$ from measurement variance $R$;
- implement the Kalman prediction and correction steps;
- interpret innovation, innovation variance, covariance and Kalman gain;
- estimate vertical velocity from noisy depth measurements;
- inspect uncertainty bands and estimation error;
- investigate how assumptions about $Q$ and $R$ change filter behavior;
- explain the relationship between a Kalman filter and a Luenberger observer.

## From Notebook 07 to Notebook 08

Notebook 07 used a Luenberger observer:

$$
\dot{\hat{\boldsymbol{\xi}}}
=
A\hat{\boldsymbol{\xi}}
+L(y-C\hat{\boldsymbol{\xi}}).
$$

Its gain $L$ was selected through pole placement. The Kalman filter keeps the predict–compare–correct structure but calculates a discrete gain from propagated state uncertainty and assumed measurement uncertainty.

## Model conventions and assumptions

- The state is $\boldsymbol{\xi}=[\delta z,\delta v]^{\mathsf T}$.
- Depth is positive downward; velocity is positive upward.
- The plant is the local linear model around $20\ \mathrm{m}$.
- Samples are equally spaced by $\Delta t=0.1\ \mathrm{s}$.
- Process and measurement noises are independent, zero-mean and Gaussian.
- Their covariances are known in the baseline experiment.
- Depth deviation is the only measured output.
- Bias, dropout, nonlinear motion and parameter drift are omitted.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

## Continuous local model

Reuse the canonical Chapter 6 operating state and compute

$$
A=
\begin{bmatrix}
0&-1\\
a&0
\end{bmatrix},
\qquad
C=
\begin{bmatrix}
1&0
\end{bmatrix}.
$$

In [ ]:
rho = 1025.0          # seawater density [kg/m^3]
g = 9.80665           # gravitational acceleration [m/s^2]
p0 = 101_325.0        # surface absolute pressure [Pa]
mass = 85.0           # diver-and-equipment mass [kg]
equilibrium_depth = 20.0  # [m]
surface_gas_volume = 8.0e-3  # [m^3]

pressure_at_equilibrium = p0 + rho * g * equilibrium_depth
k_b = (
    -(rho * g) ** 2
    * surface_gas_volume
    * p0
    / pressure_at_equilibrium**2
)
a = k_b / mass

A = np.array([
    [0.0, -1.0],
    [a, 0.0],
])
C = np.array([[1.0, 0.0]])

print("A =")
print(A)
print("C =")
print(C)

## Discretize the model

For sample period $\Delta t$,

$$
F=e^{A\Delta t}.
$$

We compare the exact matrix exponential with the first-order approximation $I+A\Delta t$.

In [ ]:
sample_period = 0.1  # [s]
F = expm(A * sample_period)
F_first_order = np.eye(2) + A * sample_period

print("Exact transition F:")
print(F)
print("First-order approximation:")
print(F_first_order)
print("Maximum elementwise difference:", np.max(np.abs(F - F_first_order)))

The exact transition is used below. Discretization and covariance construction must refer to the same sampling interval.

## Stochastic model

The synthetic truth follows

$$
\boldsymbol{\xi}_{k+1}=F\boldsymbol{\xi}_k+\mathbf{w}_k,
\qquad
\mathbf{w}_k\sim\mathcal{N}(0,Q),
$$

and the measurement follows

$$
y_k=C\boldsymbol{\xi}_k+\nu_k,
\qquad
\nu_k\sim\mathcal{N}(0,R).
$$

$Q$ is a covariance matrix for one discrete transition. $R$ is the scalar depth-measurement variance.

In [ ]:
process_covariance_true = np.diag([
    (0.0010) ** 2,  # depth-state increment variance [m^2]
    (0.0030) ** 2,  # velocity-state increment variance [(m/s)^2]
])
measurement_noise_std = 0.03  # [m]
measurement_variance_true = measurement_noise_std**2

print("True process covariance Q:")
print(process_covariance_true)
print(f"True measurement variance R: {measurement_variance_true:.6f} m^2")

## Simulate a stochastic true trajectory

In [ ]:
duration = 30.0  # [s]
sample_count = int(round(duration / sample_period)) + 1
sample_times = np.linspace(0.0, duration, sample_count)

rng = np.random.default_rng(42)
true_state = np.zeros((sample_count, 2))
true_state[0] = np.array([0.0, 0.03])

for index in range(1, sample_count):
    process_noise = rng.multivariate_normal(
        mean=np.zeros(2),
        cov=process_covariance_true,
    )
    true_state[index] = F @ true_state[index - 1] + process_noise

measurement_noise = rng.normal(
    0.0,
    measurement_noise_std,
    size=sample_count,
)
measured_depth = true_state[:, 0] + measurement_noise

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True)

axes[0].plot(sample_times, true_state[:, 0], label="true depth deviation")
axes[0].plot(
    sample_times,
    measured_depth,
    alpha=0.45,
    linewidth=0.9,
    label="noisy derived depth",
)
axes[0].set_ylabel("Depth deviation [m]")
axes[0].set_title("Synthetic stochastic trajectory and measured output")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(sample_times, true_state[:, 1])
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("True velocity deviation [m/s]")
axes[1].grid(True)

plt.tight_layout()
plt.show()

Velocity is part of the synthetic truth but is not passed to the filter as a measurement.

## Kalman-filter function

At each sample the filter:

1. predicts the state and covariance;
2. computes the innovation and its variance;
3. calculates the Kalman gain;
4. corrects the state and covariance.

The Joseph covariance update is used for numerical robustness.

In [ ]:
def run_kalman_filter(
    measurements,
    initial_estimate,
    initial_covariance,
    process_covariance,
    measurement_variance,
):
    count = len(measurements)
    dimension = F.shape[0]
    identity = np.eye(dimension)

    prior_state = np.zeros((count, dimension))
    posterior_state = np.zeros((count, dimension))
    prior_covariance = np.zeros((count, dimension, dimension))
    posterior_covariance = np.zeros((count, dimension, dimension))
    kalman_gain = np.zeros((count, dimension))
    innovation = np.zeros(count)
    innovation_variance = np.zeros(count)

    previous_state = np.asarray(initial_estimate, dtype=float)
    previous_covariance = np.asarray(initial_covariance, dtype=float)

    for index, measurement in enumerate(measurements):
        if index == 0:
            predicted_state = previous_state
            predicted_covariance = previous_covariance
        else:
            predicted_state = F @ previous_state
            predicted_covariance = (
                F @ previous_covariance @ F.T
                + process_covariance
            )

        residual = measurement - float((C @ predicted_state).item())
        residual_variance = float(
            (C @ predicted_covariance @ C.T).item()
            + measurement_variance
        )
        gain = (predicted_covariance @ C.T)[:, 0] / residual_variance

        corrected_state = predicted_state + gain * residual
        correction_matrix = identity - np.outer(gain, C[0])
        corrected_covariance = (
            correction_matrix
            @ predicted_covariance
            @ correction_matrix.T
            + np.outer(gain, gain) * measurement_variance
        )
        corrected_covariance = 0.5 * (
            corrected_covariance + corrected_covariance.T
        )

        prior_state[index] = predicted_state
        posterior_state[index] = corrected_state
        prior_covariance[index] = predicted_covariance
        posterior_covariance[index] = corrected_covariance
        kalman_gain[index] = gain
        innovation[index] = residual
        innovation_variance[index] = residual_variance

        previous_state = corrected_state
        previous_covariance = corrected_covariance

    return {
        "prior_state": prior_state,
        "posterior_state": posterior_state,
        "prior_covariance": prior_covariance,
        "posterior_covariance": posterior_covariance,
        "kalman_gain": kalman_gain,
        "innovation": innovation,
        "innovation_variance": innovation_variance,
    }

## Initialize the filter

The initial estimate is deliberately wrong. The initial covariance expresses substantial uncertainty about both state components.

In [ ]:
initial_estimate = np.array([0.40, -0.10])
initial_covariance = np.diag([0.50**2, 0.20**2])

baseline_result = run_kalman_filter(
    measured_depth,
    initial_estimate,
    initial_covariance,
    process_covariance_true,
    measurement_variance_true,
)

estimated_state = baseline_result["posterior_state"]
estimated_covariance = baseline_result["posterior_covariance"]

## Estimated depth and unmeasured velocity

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True)

axes[0].plot(sample_times, true_state[:, 0], label="true depth deviation")
axes[0].plot(sample_times, estimated_state[:, 0], label="Kalman depth estimate")
axes[0].set_ylabel("Depth deviation [m]")
axes[0].set_title("Kalman state estimate")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(sample_times, true_state[:, 1], label="true velocity deviation")
axes[1].plot(sample_times, estimated_state[:, 1], label="estimated velocity deviation")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Velocity deviation [m/s]")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

## Key result

The filter reconstructs velocity without receiving a velocity measurement. Model coupling and depth–velocity error covariance allow the scalar depth innovation to correct both state components.

## Uncertainty bands

The filter's internal one-standard-deviation uncertainties are

$$
\sigma_z=\sqrt{P_{11}},
\qquad
\sigma_v=\sqrt{P_{22}}.
$$

Plot posterior estimates with $\pm2\sigma$ bands.

In [ ]:
state_std = np.sqrt(np.maximum(
    np.diagonal(estimated_covariance, axis1=1, axis2=2),
    0.0,
))

fig, axes = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True)
state_labels = ["Depth deviation [m]", "Velocity deviation [m/s]"]
titles = ["Depth estimate uncertainty", "Velocity estimate uncertainty"]

for state_index, axis in enumerate(axes):
    lower = estimated_state[:, state_index] - 2.0 * state_std[:, state_index]
    upper = estimated_state[:, state_index] + 2.0 * state_std[:, state_index]
    axis.fill_between(
        sample_times,
        lower,
        upper,
        alpha=0.25,
        label="posterior ±2σ",
    )
    axis.plot(
        sample_times,
        true_state[:, state_index],
        color="black",
        linewidth=1.8,
        label="true state",
    )
    axis.plot(
        sample_times,
        estimated_state[:, state_index],
        label="posterior estimate",
    )
    axis.set_ylabel(state_labels[state_index])
    axis.set_title(titles[state_index])
    axis.grid(True)
    axis.legend()

axes[1].set_xlabel("Time [s]")
plt.tight_layout()
plt.show()

The bands describe uncertainty under the filter's model and covariance assumptions. They are not guaranteed physical bounds.

## Quantify error and coverage

In [ ]:
estimation_error = true_state - estimated_state
rmse = np.sqrt(np.mean(estimation_error**2, axis=0))
inside_two_sigma = np.abs(estimation_error) <= 2.0 * state_std
coverage = np.mean(inside_two_sigma, axis=0)

print(f"Depth RMSE:    {rmse[0]:.5f} m")
print(f"Velocity RMSE: {rmse[1]:.5f} m/s")
print(f"Depth ±2σ coverage:    {coverage[0]:.3f}")
print(f"Velocity ±2σ coverage: {coverage[1]:.3f}")

Coverage need not equal exactly $95\%$ in one finite, temporally correlated realization. Large systematic disagreement would motivate checking model and covariance assumptions.

## Innovation and innovation uncertainty

The innovation is

$$
r_k=y_k-C\hat{\boldsymbol{\xi}}_{k|k-1},
$$

with standard deviation $\sqrt{S_k}$. Plot it with $\pm2\sqrt{S_k}$ bands.

In [ ]:
innovation = baseline_result["innovation"]
innovation_std = np.sqrt(baseline_result["innovation_variance"])

plt.figure(figsize=(8, 4.5))
plt.fill_between(
    sample_times,
    -2.0 * innovation_std,
    2.0 * innovation_std,
    alpha=0.25,
    label="expected ±2√S",
)
plt.plot(sample_times, innovation, linewidth=0.9, label="innovation")
plt.axhline(0.0, color="black", linewidth=1)
plt.xlabel("Time [s]")
plt.ylabel("Depth innovation [m]")
plt.title("Innovation relative to predicted uncertainty")
plt.grid(True)
plt.legend()
plt.show()

Innovation is the information carried by each new measurement after accounting for the prediction. Interpreting it requires its covariance, not only its raw magnitude.

## Watch the Kalman gain evolve

In [ ]:
kalman_gain = baseline_result["kalman_gain"]

plt.figure(figsize=(8, 4.5))
plt.plot(sample_times, kalman_gain[:, 0], label="depth gain K_z")
plt.plot(sample_times, kalman_gain[:, 1], label="velocity gain K_v")
plt.xlabel("Time [s]")
plt.ylabel("Kalman gain component")
plt.title("Time-varying correction gain")
plt.grid(True)
plt.legend()
plt.show()

The velocity gain is nonzero even though the measurement contains depth only. The gain approaches a near-steady value as the covariance recursion settles under time-invariant assumptions.

## Watch posterior uncertainty evolve

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(sample_times, state_std[:, 0], label="depth standard deviation [m]")
plt.plot(sample_times, state_std[:, 1], label="velocity standard deviation [m/s]")
plt.xlabel("Time [s]")
plt.ylabel("Posterior standard deviation")
plt.title("Filter uncertainty after each correction")
plt.grid(True)
plt.legend()
plt.show()

Large initial uncertainty contracts rapidly as measurements arrive. Process noise prevents uncertainty from collapsing to zero.

## Experiment 1 — assumed measurement uncertainty

Run the same measurements through filters that assume different values of $R$. Only the filter assumption changes; the synthetic data remain fixed.

In [ ]:
measurement_assumptions = {
    "R one-quarter of true": 0.25 * measurement_variance_true,
    "R equal to true": measurement_variance_true,
    "R four times true": 4.0 * measurement_variance_true,
}

measurement_results = {}
for label, assumed_R in measurement_assumptions.items():
    result = run_kalman_filter(
        measured_depth,
        initial_estimate,
        initial_covariance,
        process_covariance_true,
        assumed_R,
    )
    velocity_rmse = np.sqrt(np.mean(
        (true_state[:, 1] - result["posterior_state"][:, 1]) ** 2
    ))
    measurement_results[label] = (result, velocity_rmse)
    print(f"{label}: velocity RMSE = {velocity_rmse:.5f} m/s")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    sample_times,
    true_state[:, 1],
    color="black",
    linewidth=2.4,
    label="true velocity deviation",
)

for label, (result, _) in measurement_results.items():
    plt.plot(sample_times, result["posterior_state"][:, 1], label=label)

plt.xlabel("Time [s]")
plt.ylabel("Velocity deviation [m/s]")
plt.title("Effect of assumed measurement variance")
plt.grid(True)
plt.legend()
plt.show()

Underestimating $R$ makes the filter react more strongly to noisy depth samples. Overestimating $R$ makes it rely more heavily on the model and slows correction of initial error.

## Experiment 2 — assumed process uncertainty

Scale $Q$ while keeping the same synthetic truth and measurements.

In [ ]:
process_assumptions = {
    "Q one-tenth of true": 0.1 * process_covariance_true,
    "Q equal to true": process_covariance_true,
    "Q ten times true": 10.0 * process_covariance_true,
}

process_results = {}
for label, assumed_Q in process_assumptions.items():
    result = run_kalman_filter(
        measured_depth,
        initial_estimate,
        initial_covariance,
        assumed_Q,
        measurement_variance_true,
    )
    velocity_rmse = np.sqrt(np.mean(
        (true_state[:, 1] - result["posterior_state"][:, 1]) ** 2
    ))
    process_results[label] = (result, velocity_rmse)
    print(f"{label}: velocity RMSE = {velocity_rmse:.5f} m/s")

Larger assumed $Q$ makes predicted covariance grow faster, generally increasing measurement influence. Smaller assumed $Q$ expresses stronger confidence in the process model. The best result for one random realization need not identify the true covariance; covariance selection requires repeated data and model validation.

## Kalman filter versus Luenberger observer

Both methods use model prediction and innovation correction. The Luenberger gain is chosen to place deterministic error poles. The Kalman gain is recomputed from covariance propagation and an uncertainty model.

Under time-invariant conditions, the Kalman gain often approaches a steady value. It then resembles a fixed-gain discrete observer, but its value was obtained from $Q$, $R$, and the Riccati covariance recursion rather than selected poles.

## What this filter does not solve

The baseline assumptions exclude:

- constant pressure-sensor bias;
- incorrect surface-pressure reference;
- wrong water density;
- nonlinear motion far from equilibrium;
- delayed, missing or asynchronous measurements;
- unknown covariance values;
- human and equipment control actions.

The Kalman filter does not automatically recognize these violations. A small reported covariance can coexist with a biased estimate when the model is confidently wrong.

## Exercises

### 1. Change the initial covariance

Compare a very confident and a very uncertain $P_0$ while keeping the same incorrect initial state estimate. Plot the first five seconds of the gains and errors.

In [ ]:
# Your code here

### 2. Change measurement variance

Generate new measurements with depth-noise standard deviations of $0.01$, $0.03$, and $0.08\ \mathrm{m}$. Use the matching $R$ in each filter and compare state RMSE.

In [ ]:
# Your code here

### 3. Inspect cross-covariance

Plot $P_{12}$ over time. Relate its sign to the velocity component of the Kalman gain.

In [ ]:
# Your code here

### 4. Introduce a wrong dynamic model

Run the filter with a value of $a$ that differs from the value used to generate the truth. Compare error, covariance bands and innovation behavior.

In [ ]:
# Your code here

## Challenge — pressure bias

Add a constant depth-equivalent bias to every measurement without changing the filter model. Determine whether the posterior uncertainty bands reveal the resulting estimation bias. Explain which zero-mean-noise assumption has been violated and propose a state augmentation that could represent the bias.

## Summary

- The Kalman filter propagates a state estimate and its uncertainty.
- $Q$ represents process uncertainty per discrete transition; $R$ represents measurement uncertainty.
- Prediction advances both the state and covariance.
- Innovation compares measured depth with predicted depth.
- The Kalman gain balances predicted uncertainty against measurement uncertainty.
- Cross-covariance allows depth measurements to correct unmeasured velocity.
- The Joseph form provides a robust covariance correction.
- Kalman gains can evolve toward steady values under time-invariant assumptions.
- Changing $Q$, $R$, or $P_0$ changes the filter's confidence and response.
- Optimality and uncertainty bands are meaningful only within the stated modeling assumptions.

## Next notebook

Notebook 09 will begin Part III with Laplace transforms and transfer functions. It will convert linear differential equations into algebraic input–output relations and prepare the frequency-domain tools used in control design.